# Advanced Analytics for Mutual Fund Portfolio

This notebook performs advanced quantitative analysis on mutual fund data:

1. Daily Returns Computation
2. Historical VaR (95%) & CVaR
3. Rolling 90-Day Sharpe Ratio
4. Investor Cohort Analysis
5. SIP Continuity Analysis
6. Fund Recommender
7. Sector HHI Concentration
8. Advanced Insights

Data sources: `bluestock_mf.db` (SQLite) and `data/processed/*.csv`

## Section 1: Setup

In [ ]:
import sqlite3
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_theme(style='darkgrid', palette='Set2', font_scale=1.1)
pio.templates.default = 'plotly_white'

%matplotlib inline

print('Libraries loaded successfully.')

In [ ]:
DB_PATH = Path('bluestock_mf.db')
DATA_DIR = Path('data/processed')
REPORTS_DIR = Path('reports')
CHARTS_DIR = Path('charts')

REPORTS_DIR.mkdir(exist_ok=True)
CHARTS_DIR.mkdir(exist_ok=True)

conn = sqlite3.connect(DB_PATH)

funds_df = pd.read_sql('SELECT * FROM dim_fund', conn)
print(f'Funds loaded: {len(funds_df)}')
print(f'Risk categories: {funds_df["risk_category"].unique()}')

## Section 2: Daily Returns Computation

In [ ]:
nav_query = '''
    SELECT fn.amfi_code, dd.full_date, fn.nav
    FROM fact_nav fn
    JOIN dim_date dd ON fn.date_id = dd.date_id
    ORDER BY fn.amfi_code, dd.full_date
'''

nav_df = pd.read_sql(nav_query, conn)
nav_df['full_date'] = pd.to_datetime(nav_df['full_date'])
print(f'NAV records: {len(nav_df):,}')
print(f'Date range: {nav_df["full_date"].min()} to {nav_df["full_date"].max()}')
print(f'Unique funds: {nav_df["amfi_code"].nunique()}')
nav_df.head()

In [ ]:
nav_pivot = nav_df.pivot_table(
    index='full_date',
    columns='amfi_code',
    values='nav'
).sort_index()

nav_pivot = nav_pivot.ffill()
print(f'Pivot shape (dates x funds): {nav_pivot.shape}')
nav_pivot.head()

In [ ]:
daily_returns = nav_pivot.pct_change().dropna(how='all')
daily_returns = daily_returns.iloc[1:]

print(f'Daily returns shape: {daily_returns.shape}')
print(f'Date range: {daily_returns.index[0].date()} to {daily_returns.index[-1].date()}')
daily_returns.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sample_funds = daily_returns.columns[:5]
for code in sample_funds:
    cum_ret = (1 + daily_returns[code]).cumprod() - 1
    ax.plot(daily_returns.index, cum_ret, label=f'Fund {code}', alpha=0.8)
ax.set_title('Cumulative Returns - Sample of 5 Funds')
ax.set_xlabel('Date'); ax.set_ylabel('Cumulative Return')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## Section 3: Historical VaR (95%) & CVaR

In [ ]:
var_95 = daily_returns.quantile(0.05)

cvar_95 = {}
for col in daily_returns.columns:
    rets = daily_returns[col].dropna()
    threshold = rets.quantile(0.05)
    below_var = rets[rets <= threshold]
    cvar_95[col] = below_var.mean() if len(below_var) > 0 else np.nan
cvar_95 = pd.Series(cvar_95)

var_cvar_df = pd.DataFrame({
    'VaR_95_pct': var_95 * 100,
    'CVaR_95_pct': cvar_95 * 100,
    'n_observations': daily_returns.count()
}).sort_values('VaR_95_pct', ascending=False)

var_cvar_df = var_cvar_df.merge(
    funds_df[['amfi_code', 'scheme_name', 'risk_category']],
    left_index=True, right_on='amfi_code', how='left'
).set_index('amfi_code')

style = var_cvar_df.style \
    .format({'VaR_95_pct': '{:.3f}', 'CVaR_95_pct': '{:.3f}'}) \
    .background_gradient(subset=['VaR_95_pct'], cmap='RdYlGn') \
    .set_caption('Funds ranked by VaR (best = least negative)')
style

In [ ]:
var_cvar_df.to_csv(REPORTS_DIR / 'var_cvar_report.csv')
print(f'Saved var_cvar_report.csv to {REPORTS_DIR}')

In [ ]:
plot_df = var_cvar_df.sort_values('VaR_95_pct', ascending=True).head(25)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d7191c' if v < -3 else '#fdae61' if v < -2 else '#a6d96a' for v in plot_df['VaR_95_pct']]
bars = ax.barh(plot_df['scheme_name'].str[:50], plot_df['VaR_95_pct'], color=colors, edgecolor='white')
ax.axvline(x=plot_df['VaR_95_pct'].median(), color='black', linestyle='--', linewidth=1, label='Median')
ax.set_xlabel('VaR 95% (%)'); ax.set_title('Historical 95% VaR by Fund (Most Negative at Top)')
ax.invert_yaxis(); ax.legend(); plt.tight_layout()
plt.savefig(CHARTS_DIR / 'var_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4: Rolling 90-Day Sharpe Ratio

In [ ]:
key_funds = {
    119551: 'SBI Bluechip - Regular',
    100016: 'HDFC Top 100 - Regular',
    120503: 'ICICI Pru Bluechip - Regular',
    118632: 'Nippon Large Cap - Regular',
    119092: 'Axis Bluechip - Regular'
}

available_codes = [c for c in key_funds if c in daily_returns.columns]
print(f'Available key funds: {len(available_codes)} / {len(key_funds)}')
for c in available_codes:
    print(f'  {c}: {key_funds[c]}')

In [ ]:
WINDOW = 90
ANNUAL_FACTOR = np.sqrt(252)

rolling_sharpe = {}
for code in available_codes:
    rets = daily_returns[code].dropna()
    roll_mean = rets.rolling(WINDOW).mean()
    roll_std = rets.rolling(WINDOW).std()
    rolling_sharpe[code] = (roll_mean / roll_std) * ANNUAL_FACTOR
    print(f'{key_funds[code]}: {rolling_sharpe[code].notna().sum()} rolling points')

In [ ]:
fig = go.Figure()
for code, label in key_funds.items():
    if code in rolling_sharpe:
        fig.add_trace(go.Scatter(
            x=rolling_sharpe[code].index,
            y=rolling_sharpe[code].values,
            mode='lines',
            name=label
        ))

fig.update_layout(
    title='Rolling 90-Day Sharpe Ratio - 5 Key Large Cap Funds',
    xaxis_title='Date',
    yaxis_title='Sharpe Ratio (Annualised)',
    height=500, width=1000,
    hovermode='x unified',
    legend=dict(orientation='h', y=1.12)
)

fig.write_image(CHARTS_DIR / 'rolling_sharpe.png')
fig.show()

## Section 5: Investor Cohort Analysis

In [ ]:
txn_query = '''
    SELECT transaction_id, amfi_code, transaction_date, amount_inr,
           transaction_type, state, city, city_tier, gender, annual_income_lakh
    FROM fact_transactions
'''

txn_df = pd.read_sql(txn_query, conn)
txn_df['transaction_date'] = pd.to_datetime(txn_df['transaction_date'])
txn_df['transaction_year'] = txn_df['transaction_date'].dt.year

print(f'Transactions loaded: {len(txn_df):,}')
print(f'Unique investors (transaction_id proxy): {txn_df["transaction_id"].nunique():,}')
txn_df.head()

In [ ]:
investor_first_year = txn_df.groupby('transaction_id')['transaction_year'].min().reset_index()
investor_first_year.columns = ['transaction_id', 'cohort_year']

txn_with_cohort = txn_df.merge(investor_first_year, on='transaction_id')

cohort_stats = txn_with_cohort.groupby('cohort_year').agg(
    investor_count=('transaction_id', 'nunique'),
    total_amount=('amount_inr', 'sum'),
    avg_amount_per_txn=('amount_inr', 'mean'),
    median_income=('annual_income_lakh', 'median')
).round(2)

top_fund_by_cohort = {}
for year, grp in txn_with_cohort.groupby('cohort_year'):
    top_code = grp['amfi_code'].value_counts().index[0]
    top_name = funds_df[funds_df['amfi_code'] == top_code]['scheme_name'].values
    top_fund_by_cohort[year] = top_name[0] if len(top_name) > 0 else str(top_code)

cohort_stats['top_fund_preference'] = pd.Series(top_fund_by_cohort)

display(cohort_stats.style
         .format({'total_amount': '{:,.0f}', 'avg_amount_per_txn': '{:,.0f}', 'median_income': '{:.1f}'})
         .set_caption('Investor Cohort Analysis'))

## Section 6: SIP Continuity Analysis

In [ ]:
sip_df = txn_df[txn_df['transaction_type'] == 'SIP'].copy()
sip_df = sip_df.sort_values(['transaction_id', 'transaction_date'])

sip_counts = sip_df.groupby('transaction_id').size()
regular_sip_ids = sip_counts[sip_counts >= 6].index
sip_regular = sip_df[sip_df['transaction_id'].isin(regular_sip_ids)].copy()

print(f'Total SIP investors: {sip_df["transaction_id"].nunique():,}')
print(f'SIP investors with 6+ transactions: {len(regular_sip_ids):,}')

In [ ]:
sip_regular['prev_date'] = sip_regular.groupby('transaction_id')['transaction_date'].shift(1)
sip_regular['gap_days'] = (sip_regular['transaction_date'] - sip_regular['prev_date']).dt.days

investor_gaps = sip_regular.groupby('transaction_id')['gap_days'].agg(['mean', 'std', 'count']).dropna()
investor_gaps.columns = ['avg_gap_days', 'std_gap_days', 'num_gaps']

investor_gaps['at_risk'] = investor_gaps['avg_gap_days'] > 35

total_sip = len(investor_gaps)
continuous = len(investor_gaps[~investor_gaps['at_risk']])
at_risk = len(investor_gaps[investor_gaps['at_risk']])
at_risk_pct = (at_risk / total_sip * 100) if total_sip > 0 else 0

print(f'Total SIP investors with sufficient history: {total_sip:,}')
print(f'Continuous investors: {continuous:,} ({continuous/total_sip*100:.1f}%)')
print(f'At-risk investors (avg gap > 35 days): {at_risk:,} ({at_risk_pct:.1f}%)')
print(f'\nGap statistics:')
print(investor_gaps['avg_gap_days'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(investor_gaps['avg_gap_days'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(x=35, color='red', linestyle='--', linewidth=2, label='At-Risk Threshold (35d)')
axes[0].axvline(x=30, color='orange', linestyle='--', linewidth=1.5, label='SIP Ideal (30d)')
axes[0].set_xlabel('Average Gap (days)'); axes[0].set_ylabel('Number of Investors')
axes[0].set_title('Distribution of Average SIP Gaps'); axes[0].legend()

at_risk_gaps = investor_gaps[investor_gaps['at_risk']]['avg_gap_days']
axes[1].hist(at_risk_gaps, bins=30, color='salmon', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Average Gap (days)'); axes[1].set_ylabel('Count')
axes[1].set_title(f'At-Risk Investors Only (n={at_risk:,})')

plt.tight_layout(); plt.savefig(CHARTS_DIR / 'sip_gap_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

box_data = [
    investor_gaps[~investor_gaps['at_risk']]['avg_gap_days'].dropna(),
    investor_gaps[investor_gaps['at_risk']]['avg_gap_days'].dropna()
]
bp = ax.boxplot(box_data, labels=['Continuous', 'At-Risk'], patch_artist=True)
bp['boxes'][0].set_facecolor('#a6d96a'); bp['boxes'][1].set_facecolor('#d7191c')
ax.set_ylabel('Average Gap (days)'); ax.set_title('SIP Gap Comparison: Continuous vs At-Risk Investors')
ax.axhline(y=35, color='black', linestyle='--', alpha=0.5, label='35-day threshold')
ax.legend(); plt.tight_layout(); plt.show()

## Section 7: Fund Recommender

In [ ]:
perf_df = pd.read_sql('SELECT * FROM fact_performance', conn)
perf_merged = perf_df.merge(
    funds_df[['amfi_code', 'risk_category', 'sub_category', 'fund_manager']],
    on='amfi_code', how='left'
)
print(f'Performance records: {len(perf_merged)}')
perf_merged[['scheme_name', 'risk_category', 'sharpe_ratio', 'return_1yr_pct']].head()

In [ ]:
RISK_MAP = {
    'Low': ['Low'],
    'Moderate': ['Moderate', 'Moderately High'],
    'High': ['High', 'Very High']
}


def recommend_funds(risk_appetite, top_n=3):
    matching_risks = RISK_MAP.get(risk_appetite)
    if matching_risks is None:
        raise ValueError(f"Invalid risk_appetite '{risk_appetite}'. Choose: {list(RISK_MAP.keys())}")

    candidates = perf_merged[perf_merged['risk_category'].isin(matching_risks)].copy()

    if candidates.empty:
        print(f'No funds found for risk appetite: {risk_appetite}')
        return pd.DataFrame()

    candidates = candidates.sort_values('sharpe_ratio', ascending=False)
    top_funds = candidates.head(top_n)

    result_cols = [
        'amfi_code', 'scheme_name', 'fund_house', 'risk_category',
        'sharpe_ratio', 'return_1yr_pct', 'return_3yr_pct', 
        'return_5yr_pct', 'std_dev_ann_pct', 'max_drawdown_pct',
        'expense_ratio_pct', 'morningstar_rating'
    ]
    result_cols = [c for c in result_cols if c in top_funds.columns]
    return top_funds[result_cols].reset_index(drop=True)


print('Recommender function defined. Testing for all risk levels...\n')

for risk in ['Low', 'Moderate', 'High']:
    print(f'\n=== Risk Appetite: {risk} ===')
    recs = recommend_funds(risk, top_n=3)
    if not recs.empty:
        display(recs.style.format({
            'sharpe_ratio': '{:.3f}', 'return_1yr_pct': '{:.2f}',
            'return_3yr_pct': '{:.2f}', 'return_5yr_pct': '{:.2f}',
            'std_dev_ann_pct': '{:.2f}', 'max_drawdown_pct': '{:.2f}',
            'expense_ratio_pct': '{:.2f}'
        }))

### Export recommender to scripts/recommender.py

In [ ]:
recommender_code = '''
"""
Fund Recommender - Reusable module for fund recommendations by risk appetite.

Usage:
    from scripts.recommender import FundRecommender
    rec = FundRecommender('bluestock_mf.db')
    print(rec.recommend('Moderate', top_n=3))
    print(rec.recommend('High', top_n=5))
    print(rec.recommend('Low'))
"""

import sqlite3
from pathlib import Path

import pandas as pd


RISK_MAP = {
    'Low': ['Low'],
    'Moderate': ['Moderate', 'Moderately High'],
    'High': ['High', 'Very High']
}


class FundRecommender:

    def __init__(self, db_path='bluestock_mf.db'):
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path)
        self._load_data()

    def _load_data(self):
        perf_df = pd.read_sql('SELECT * FROM fact_performance', self.conn)
        funds_df = pd.read_sql('SELECT amfi_code, risk_category FROM dim_fund', self.conn)
        self.merged = perf_df.merge(funds_df, on='amfi_code', how='left')

    def recommend(self, risk_appetite, top_n=3):
        matching_risks = RISK_MAP.get(risk_appetite)
        if matching_risks is None:
            raise ValueError(
                f"Invalid risk_appetite '{risk_appetite}'. "
                f"Choose from: {list(RISK_MAP.keys())}"
            )

        candidates = self.merged[
            self.merged['risk_category'].isin(matching_risks)
        ].copy()

        candidates = candidates.sort_values('sharpe_ratio', ascending=False)
        top_funds = candidates.head(top_n)

        result_cols = [
            'amfi_code', 'scheme_name', 'fund_house', 'risk_category',
            'sharpe_ratio', 'return_1yr_pct', 'return_3yr_pct',
            'return_5yr_pct', 'std_dev_ann_pct', 'max_drawdown_pct',
            'expense_ratio_pct', 'morningstar_rating'
        ]
        result_cols = [c for c in result_cols if c in top_funds.columns]
        return top_funds[result_cols].reset_index(drop=True)

    def close(self):
        self.conn.close()


if __name__ == '__main__':
    import sys

    db_path = sys.argv[1] if len(sys.argv) > 1 else 'bluestock_mf.db'
    recommender = FundRecommender(db_path)

    for risk_level in ['Low', 'Moderate', 'High']:
        print(f'\\n=== Top 3 Funds for {risk_level} Risk ===')
        recs = recommender.recommend(risk_level)
        print(recs.to_string(index=False))

    recommender.close()
'''

scripts_dir = Path('scripts')
scripts_dir.mkdir(exist_ok=True)
with open(scripts_dir / 'recommender.py', 'w') as f:
    f.write(recommender_code)

print('Saved scripts/recommender.py')

In [ ]:
from scripts.recommender import FundRecommender

rec_cmd = FundRecommender('bluestock_mf.db')

for risk_level in ['Low', 'Moderate', 'High']:
    print(f'\n=== Top 3 Funds for {risk_level} Risk (via module) ===')
    display(rec_cmd.recommend(risk_level, top_n=3))

rec_cmd.close()

## Section 8: Sector HHI Concentration

In [ ]:
holdings_df = pd.read_csv(DATA_DIR / '09_portfolio_holdings.csv')
print(f'Portfolio holdings loaded: {len(holdings_df):,} rows')
print(f'Columns: {list(holdings_df.columns)}')
print(f'Unique funds: {holdings_df["amfi_code"].nunique()}')

holdings_df['weight_sq'] = (holdings_df['weight_pct'] / 100) ** 2

hhi = holdings_df.groupby('amfi_code')['weight_sq'].sum() * 10000
hhi = hhi.sort_values(ascending=False)

print(f'\nHHI Summary:')
print(f'  Mean HHI: {hhi.mean():.0f}')
print(f'  Median HHI: {hhi.median():.0f}')
print(f'  Min HHI: {hhi.min():.0f}')
print(f'  Max HHI: {hhi.max():.0f}')

In [ ]:
def classify_hhi(val):
    if val > 2500:
        return 'Concentrated'
    elif val >= 1500:
        return 'Moderately Concentrated'
    else:
        return 'Diversified'


hhi_classified = pd.DataFrame({
    'amfi_code': hhi.index,
    'HHI': hhi.values,
    'classification': [classify_hhi(v) for v in hhi.values]
})

hhi_classified = hhi_classified.merge(
    funds_df[['amfi_code', 'scheme_name', 'category', 'risk_category']],
    on='amfi_code', how='left'
)

print('\nClassification distribution:')
print(hhi_classified['classification'].value_counts())

print('\nMost Concentrated Funds:')
display(hhi_classified.head(5)[['scheme_name', 'HHI', 'classification', 'category']])

print('\nMost Diversified Funds:')
display(hhi_classified.tail(5)[['scheme_name', 'HHI', 'classification', 'category']])

In [ ]:
plot_hhi = hhi_classified.head(15).sort_values('HHI', ascending=True)

color_map = {
    'Concentrated': '#d7191c',
    'Moderately Concentrated': '#fdae61',
    'Diversified': '#1a9641'
}
bar_colors = [color_map[c] for c in plot_hhi['classification']]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(plot_hhi['scheme_name'].str[:45], plot_hhi['HHI'], color=bar_colors, edgecolor='white')
ax.axvline(x=2500, color='#d7191c', linestyle='--', linewidth=1.5, alpha=0.7, label='Concentrated (2500)')
ax.axvline(x=1500, color='#fdae61', linestyle='--', linewidth=1.5, alpha=0.7, label='Moderate (1500)')
ax.set_xlabel('Herfindahl-Hirschman Index (HHI)')
ax.set_title('Sector Concentration (HHI) - Top 15 Funds')

from matplotlib.patches import Patch
legend_patches = [
    Patch(color='#d7191c', label='Concentrated (>2500)'),
    Patch(color='#fdae61', label='Moderate (1500-2500)'),
    Patch(color='#1a9641', label='Diversified (<1500)')
]
ax.legend(handles=legend_patches, loc='lower right')

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'hhi_concentration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sector_hhi = holdings_df.groupby(['amfi_code', 'sector'])['weight_pct'].sum().reset_index()
sector_hhi['weight_sq'] = (sector_hhi['weight_pct'] / 100) ** 2
sector_level_hhi = sector_hhi.groupby('amfi_code')['weight_sq'].sum() * 10000

top_sector_weights = sector_hhi.loc[
    sector_hhi.groupby('amfi_code')['weight_pct'].idxmax()
][['amfi_code', 'sector', 'weight_pct']]

concentration_summary = hhi_classified[['amfi_code', 'scheme_name', 'HHI', 'classification']].merge(
    top_sector_weights, on='amfi_code', how='left'
).sort_values('HHI', ascending=False)

print('Concentration Summary (with dominant sector):')
display(concentration_summary.head(10).style
         .format({'HHI': '{:.0f}', 'weight_pct': '{:.1f}'})
         .background_gradient(subset=['HHI'], cmap='Reds'))

## Section 9: 5 Advanced Insights

### Insight 1: Funds with Highest VaR (Most Downside Risk)

The Historical Value at Risk (95% VaR) analysis identifies funds with the greatest potential for daily losses. 

In [ ]:
print('=== Top 5 Funds by VaR - Most Downside Risk ===')
top_risky = var_cvar_df.sort_values('VaR_95_pct', ascending=True).head(5)
display(top_risky[['scheme_name', 'risk_category', 'VaR_95_pct', 'CVaR_95_pct']].style
        .format({'VaR_95_pct': '{:.3f}', 'CVaR_95_pct': '{:.3f}'})
        .set_caption('Funds with most negative VaR'))

print(f'\nWorst VaR: {top_risky["VaR_95_pct"].iloc[0]:.3f}% daily ({top_risky["scheme_name"].iloc[0][:50]})')
print(f'Best VaR: {var_cvar_df["VaR_95_pct"].max():.3f}% daily ({var_cvar_df.loc[var_cvar_df["VaR_95_pct"].idxmax(), "scheme_name"][:50]})')
print(f'Spread: {var_cvar_df["VaR_95_pct"].max() - var_cvar_df["VaR_95_pct"].min():.3f} percentage points')

### Insight 2: Investor Cohort with Highest Investment Levels

Cohort analysis reveals which vintage of investors contributes most to total AUM. Understanding cohort behavior helps target retention campaigns and predict future redemptions.

In [ ]:
top_cohort = cohort_stats.sort_values('total_amount', ascending=False).head(3)
print('Top 3 Investor Cohorts by Total Invested Amount:')
display(top_cohort[['investor_count', 'total_amount', 'avg_amount_per_txn', 'top_fund_preference']].style
        .format({'total_amount': '{:,.0f}', 'avg_amount_per_txn': '{:,.0f}'}))

largest_cohort = cohort_stats['total_amount'].idxmax()
print(f'\nLargest cohort: {largest_cohort} with '
      f'{cohort_stats.loc[largest_cohort, "investor_count"]:,} investors '
      f'investing INR {cohort_stats.loc[largest_cohort, "total_amount"]:,.0f}')

### Insight 3: SIP Continuity Rate and Business Implications

SIP continuity is a leading indicator of investor stickiness and long-term AUM growth. A high at-risk rate signals potential redemption pressure and the need for proactive investor engagement.

In [ ]:
print('=== SIP Continuity Summary ===')
print(f'Total SIP investors (6+ transactions): {total_sip:,}')
print(f'Continuous: {continuous:,} ({continuous/total_sip*100:.1f}%)')
print(f'At-Risk: {at_risk:,} ({at_risk_pct:.1f}%)')
print(f'\nMedian avg gap (continuous): {investor_gaps[~investor_gaps["at_risk"]]["avg_gap_days"].median():.1f} days')
print(f'Median avg gap (at-risk): {investor_gaps[investor_gaps["at_risk"]]["avg_gap_days"].median():.1f} days')

if at_risk_pct > 20:
    print('\n⚠️  WARNING: High at-risk rate. Consider automated SIP reminder campaigns.')
else:
    print('\n✅ SIP continuity rate is healthy. Most investors maintain regular contributions.')

### Insight 4: Funds with Most Concentrated Sector Bets (Highest HHI)

The Herfindahl-Hirschman Index measures portfolio concentration. Higher HHI values indicate concentrated sector bets, which can amplify both returns and risk.

In [ ]:
print('=== Sector HHI Concentration Insights ===')
print(f'Funds above 2500 HHI (Concentrated): {len(hhi_classified[hhi_classified["classification"] == "Concentrated"])}')
print(f'Funds 1500-2500 HHI (Moderate): {len(hhi_classified[hhi_classified["classification"] == "Moderately Concentrated"])}')
print(f'Funds below 1500 HHI (Diversified): {len(hhi_classified[hhi_classified["classification"] == "Diversified"])}')

concentrated = hhi_classified[hhi_classified['classification'] == 'Concentrated']
print(f'\nMost Concentrated: {concentrated.iloc[0]["scheme_name"][:60]} (HHI: {concentrated.iloc[0]["HHI"]:.0f})')
print(f'   Dominant sector: {concentration_summary.loc[concentration_summary["amfi_code"] == concentrated.iloc[0]["amfi_code"], "sector"].values[0]} '
      f'({concentration_summary.loc[concentration_summary["amfi_code"] == concentrated.iloc[0]["amfi_code"], "weight_pct"].values[0]:.1f}%)')

diversified = hhi_classified[hhi_classified['classification'] == 'Diversified'].sort_values('HHI')
print(f'\nMost Diversified: {diversified.iloc[0]["scheme_name"][:60]} (HHI: {diversified.iloc[0]["HHI"]:.0f})')

### Insight 5: Best Fund Recommendations by Risk Profile

Using the risk-to-performance mapping, the recommender identifies the top-performing funds for each risk appetite based on the Sharpe ratio, balancing returns against volatility.

In [ ]:
print('=== Recommended Funds by Risk Profile ===\n')

risk_summary = []
for risk_level in ['Low', 'Moderate', 'High']:
    recs = recommend_funds(risk_level, top_n=3)
    if not recs.empty:
        top = recs.iloc[0]
        risk_summary.append({
            'Risk Profile': risk_level,
            'Top Pick': top['scheme_name'][:50],
            'Sharpe': f'{top["sharpe_ratio"]:.2f}',
            '1Y Return': f'{top["return_1yr_pct"]:.1f}%',
            '3Y Return': f'{top["return_3yr_pct"]:.1f}%' if 'return_3yr_pct' in recs.columns else 'N/A',
            'Max DD': f'{top["max_drawdown_pct"]:.1f}%' if 'max_drawdown_pct' in recs.columns else 'N/A',
            'Expense': f'{top["expense_ratio_pct"]:.2f}%' if 'expense_ratio_pct' in recs.columns else 'N/A'
        })

risk_summary_df = pd.DataFrame(risk_summary)
display(risk_summary_df)

print('\n✅ Recommendations are based on highest Sharpe Ratio within matching risk categories.')
print('   Investors with Low risk appetite are matched to Low-risk funds (debt/gilt).')
print('   Investors with Moderate risk matched to Moderate/Moderately High (large cap).')
print('   Investors with High risk matched to High/Very High (small cap/mid cap).')

## Bonus: Risk-Return Scatter

In [ ]:
import plotly.express as px

scatter_data = perf_merged[['scheme_name', 'risk_category', 'sharpe_ratio',
                             'return_1yr_pct', 'std_dev_ann_pct', 'aum_crore']].dropna()

fig = px.scatter(
    scatter_data,
    x='std_dev_ann_pct', y='return_1yr_pct',
    size='aum_crore', color='risk_category',
    hover_name='scheme_name',
    title='Risk vs Return: 1-Year Return vs Annualised Standard Deviation',
    labels={'std_dev_ann_pct': 'Annualised Std Dev (%)', 'return_1yr_pct': '1-Year Return (%)'},
    height=550, width=950
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.show()

In [ ]:
print('\n' + '='*60)
print('Advanced Analytics Complete')
print('='*60)
print(f'Reports exported: var_cvar_report.csv -> {REPORTS_DIR}')
print(f'Charts saved: rolling_sharpe.png, var_bar_chart.png,')
print(f'              sip_gap_distribution.png, hhi_concentration.png -> {CHARTS_DIR}')
print(f'Script exported: recommender.py -> {scripts_dir}')

conn.close()
print('\nDatabase connection closed. Notebook execution complete.')